# Testing inference

In [ ]:
import torch
from transformers import BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

In [ ]:
from transformers import pipeline

model_id = "llava-hf/llava-1.5-7b-hf"

pipe = pipeline("image-text-to-text", model=model_id, model_kwargs={"quantization_config": quantization_config})

Loading weights:   0%|          | 0/686 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

processor_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/674 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/41.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/552 [00:00<?, ?B/s]

In [ ]:
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                "image": "https://drive.google.com/uc?export=download&id=1W9OvSWj4hqSaMyzihQnADw59Zu4Vxw-c",
            },
            {
                "type": "text",
                "text": "PROMPT: Classify this image into one of the following categories: homophobic, transphobic, misogynistic, or none. Answer only with the class name."
                # "text": "Describe the meme"
                },
        ],
    }
]

outputs = pipe(
    text=messages,
    generate_kwargs={"max_new_tokens": 200}
    )

Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [ ]:
outputs[0]["generated_text"]

[{'role': 'user',
  'content': [{'type': 'image',
    'image': 'https://drive.google.com/uc?export=download&id=1W9OvSWj4hqSaMyzihQnADw59Zu4Vxw-c'},
   {'type': 'text',
    'text': 'PROMPT: Classify this image into one of the following categories: homophobic, transphobic, misogynistic, or none. Answer only with the class name.'}]},
 {'role': 'assistant', 'content': ' Misogynistic'}]

In [ ]:
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": "நீங்க எப்படி இருக்கீங்க? தமிழில் பதில் சொல்லுங்க." # How are you? Answer in Tamil
                },
        ],
    }
]

outputs = pipe(
    text=messages,
    generate_kwargs={"max_new_tokens": 200}
    )
outputs[0]["generated_text"]

Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'role': 'user',
  'content': [{'type': 'text',
    'text': 'நீங்க எப்படி இருக்கீங்க? தமிழில் பதில் சொல்லுங்க.'}]},
 {'role': 'assistant',
  'content': ' அரே ஏற்கனவே எப்படி இருக்கீங்க? அதாவது ஏற்கனவே எப்படி இருக்கீங்க? அதாவது ஏற்கனவே எப்படி இருக்கீங்க? அதாவது ஏற்கனவே எப்படி இருக்கீங்க? அதாவது'}]

In [ ]:
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": "സുഖമാണോ? മലയാളത്തിൽ ഉത്തരം പറയാമോ?" # How are you? Answer in Malayalam
                },
        ],
    }
]

outputs = pipe(
    text=messages,
    generate_kwargs={"max_new_tokens": 200}
    )
outputs[0]["generated_text"]

Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'role': 'user',
  'content': [{'type': 'text', 'text': 'സുഖമാണോ? മലയാളത്തിൽ ഉത്തരം പറയാമോ?'}]},
 {'role': 'assistant',
  'content': ' സുഖമാണ്ട് മലയാളത്തിൽ ഉത്തരം പറയാമോ? നമ്മൾ സുഖമാണ്ട് മലയാളത്തിൽ പറയുന്നുണ്ട്. nobody is perfect.'}]

# Main prediction
# Tamil

In [ ]:
import requests
import re
import csv

FOLDER_ID = "1VVHXAW5M_GMjYmxMmy43-8TometzmuIE"
OUTPUT_CSV = "test_pred_llava_tamil.csv"

In [ ]:
from google.colab import auth
from googleapiclient.discovery import build

# Authenticate your Colab session
auth.authenticate_user()

# Build the Drive API service
drive_service = build('drive', 'v3')

In [ ]:
def list_all_files_in_folder(folder_id):
    all_files = []
    page_token = None

    while True:
        # Pass the page_token to the list request
        results = drive_service.files().list(
            q=f"'{folder_id}' in parents and trashed = false",
            fields="nextPageToken, files(id, name)",
            pageSize=1000, # You can set this up to 1000
            pageToken=page_token
        ).execute()

        items = results.get('files', [])

        for item in items:
            ext = item['name'].split(".")[-1].lower()
            if ext != "csv":
                item['name'] = item['name'].split(".")[0]
                all_files.append(item)

        # Check if there's another page of results
        page_token = results.get('nextPageToken')
        if not page_token:
            break

    return all_files

# Replace with your actual public folder ID
files = list_all_files_in_folder(FOLDER_ID)

In [ ]:
len(files)

356

In [ ]:
files[0]

{'id': '1d943aXQ5YuFnI6Maa1ubWnRRCgfK7XBa', 'name': '1262'}

In [ ]:
def classify_image(file_id):
    prompt = """Analyze this image. Does it contain hateful content?

Categories:
- homophobic: anti-gay/lesbian hate
- transphobic: anti-transgender hate
- misogynistic: anti-women hate
- none: neutral, unrelated, or no hateful content present

Answer only with the category name.
"""
    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "image": f"https://drive.google.com/uc?export=download&id={file_id}",
                },
                {
                    "type": "text",
                    "text": prompt,
                },
            ],
        }
    ]
    outputs = pipe(text=messages, generate_kwargs={"max_new_tokens": 200})
    return outputs[0]["generated_text"][-1]["content"].strip()

In [ ]:
rows = []

for i, f in enumerate(files, start=1):
    file_id = f["id"]
    print(f"[{i}/{len(files)}] Processing {file_id} ...", end=" ", flush=True)
    try:
        label = classify_image(file_id)
    except Exception as e:
        label = f"ERROR: {e}"
    print(label)
    rows.append({"image_id": f["name"], "labels": label})

[1/356] Processing 1d943aXQ5YuFnI6Maa1ubWnRRCgfK7XBa ... 

Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


None
[2/356] Processing 1_n84UA6Lh85s7M9UsXR89NITIZCcBjZZ ... 

Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


None
[3/356] Processing 1b3Bfx-fLNeJuCYJeTGSZQp4_bceSf04_ ... 

...

None
[355/356] Processing 1YLeMiGXhS5HGONUKxP-HzFHfu-m0aV5g ... 

Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


None
[356/356] Processing 1bhMXVBOI53PgfmWlZodiWI7M-GBtHKtz ... 

Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


None


In [ ]:
with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["image_id", "labels"])
    writer.writeheader()
    writer.writerows(rows)

print(f"\nDone! Saved {len(rows)} rows → {OUTPUT_CSV}")


Done! Saved 356 rows → test_pred_llava.csv


# Malayalam

In [ ]:
FOLDER_ID = "1O9LnJ1jhjMY_pQRGItBFt0Y40kcHSNrD"
OUTPUT_CSV = "test_pred_llava_malayalam.csv"

files = list_all_files_in_folder(FOLDER_ID)

In [ ]:
len(files)

200

In [ ]:
rows = []

for i, f in enumerate(files, start=1):
    file_id = f["id"]
    print(f"[{i}/{len(files)}] Processing {file_id} ...", end=" ", flush=True)
    try:
        label = classify_image(file_id)
    except Exception as e:
        label = f"ERROR: {e}"
    print(label)
    rows.append({"image_id": f["name"], "labels": label})

[1/200] Processing 1PVAxa8rDcmwG1alawv0zydDepCXsko7P ... 

Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


None
[2/200] Processing 1TPDylXbewMtYDARehatcDywsxDiwKv25 ... 

Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


None
[3/200] Processing 1Q1zskB42EgaqLUrapx8KEHZ2N65YXRBC ... 

...

None
[199/200] Processing 1Ssr9uQoLqyAzZAr8JU0Lbf0TQFyZz74i ... 

Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


None
[200/200] Processing 1SFNJxs7IL_2jkN65vMQ5ELgq7kaAQ-Nf ... 

Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


None


In [ ]:
with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["image_id", "labels"])
    writer.writeheader()
    writer.writerows(rows)

print(f"\nDone! Saved {len(rows)} rows → {OUTPUT_CSV}")


Done! Saved 200 rows → test_pred_llava_malayalam.csv
